# Chapter 4 — Prompt Structuring

### The Story
Priya is back: *"Great work! Can we also detect urgency? And handle Hindi emails? And add billing sub-categories?"*

Your current prompt is 12 lines. Adding all this = wall of text.
You need to structure prompts like you structure code.

---

### 3 Tasks
- **Task 1:** Test structured prompt on Hindi emails
- **Task 2:** Add `language` field to JSON output
- **Task 3:** Remove RULES section — see what breaks

## Setup

In [1]:
from openai import OpenAI
import json
import time
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

client = OpenAI()

with open("test_emails.json", "r") as f:
    emails = json.load(f)

print(f"Emails loaded: {len(emails)}")

Emails loaded: 100


---
## The Structured Prompt
### Before we start tasks — understand the structure

In [2]:
# This is the V5 structured prompt from Chapter 4
# Notice: sections are labeled, rules are explicit, output is JSON

STRUCTURED_PROMPT = """CRITICAL: Return ONLY valid JSON. No explanation. No markdown.

# ROLE
You are an expert support email classifier for a B2B SaaS company.
You have 10 years of experience triaging customer support emails.

# TASK
Classify the incoming email into exactly one primary category,
one urgency level, and one billing sub-category where applicable.

# CATEGORIES
Primary (pick ONE):
- Billing
- Technical
- Feature Request
- Spam
- Other

Urgency (pick ONE):
- High    (customer cannot use the product)
- Medium  (issue exists but workaround available)
- Low     (question or minor inconvenience)

Billing Sub-category (only if Primary = Billing, else Not Applicable):
- Refund Request
- Failed Payment
- Pricing Question
- Invoice Issue
- Not Applicable

# RULES
- RETURN ONLY valid JSON. No explanation. No markdown fences.
- If email is in any language, still classify correctly.
- If unsure, use Other + Low urgency.

# OUTPUT FORMAT
Return ONLY this JSON:
{{"category": "...", "urgency": "...", "billing_sub": "..."}}

# EMAIL TO CLASSIFY
Subject : {subject}
From    : {sender}
Body    : {body}"""


def classify_structured(email):
    prompt = STRUCTURED_PROMPT.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


# Quick test
email = emails[0]
result = classify_structured(email)
print(f"Email  : {email['subject']}")
print(f"Output : {result}")
print(f"Parsed : {json.loads(result)}")

Email  : Invoice Discrepancy
Output : {"category": "Billing", "urgency": "Medium", "billing_sub": "Invoice Issue"}
Parsed : {'category': 'Billing', 'urgency': 'Medium', 'billing_sub': 'Invoice Issue'}


---
## Task 1 — Test on Hindi + English Mixed Emails

In [3]:
# 3 test emails for this task
test_emails = [
    {
        "subject": "Payment refund request",
        "sender": "rahul@startup.in",
        "body": "Hi, mujhe last month ka refund chahiye. "
                "Mera subscription cancel ho gaya tha but amount deduct ho gaya.",
        "correct_label": "Billing"
    },
    {
        "subject": "App crash ho raha hai",
        "sender": "priya@company.com",
        "body": "Bhai app baar baar crash ho raha hai Settings mein jaate hi. "
                "Please fix karo urgently — demo hai kal.",
        "correct_label": "Technical"
    },
    {
        "subject": "Billing mein koi samasya hai",
        "sender": "user@firm.co",
        "body": "Mujhe do baar charge kiya gaya hai is mahine. "
                "Kripya check karein aur refund karein.",
        "correct_label": "Billing"
    }
]

print("=" * 60)
print("HINDI + ENGLISH MIXED EMAILS")
print("=" * 60)

for email in test_emails:
    result_raw = classify_structured(email)
    result = json.loads(result_raw)
    is_correct = result['category'] == email['correct_label']
    status = "✅" if is_correct else "❌"
    print(f"\n{status} Subject  : {email['subject']}")
    print(f"   Expected : {email['correct_label']}")
    print(f"   Category : {result['category']}")
    print(f"   Urgency  : {result['urgency']}")
    print(f"   Billing  : {result['billing_sub']}")
    time.sleep(1)

print()
print("The RULES section says: 'If email is in any language, still classify.'")
print("That one line handles Hindi, Tamil, Spanish — automatically.")

HINDI + ENGLISH MIXED EMAILS

✅ Subject  : Payment refund request
   Expected : Billing
   Category : Billing
   Urgency  : High
   Billing  : Refund Request

✅ Subject  : App crash ho raha hai
   Expected : Technical
   Category : Technical
   Urgency  : High
   Billing  : Not Applicable

✅ Subject  : Billing mein koi samasya hai
   Expected : Billing
   Category : Billing
   Urgency  : High
   Billing  : Refund Request

The RULES section says: 'If email is in any language, still classify.'
That one line handles Hindi, Tamil, Spanish — automatically.


In [4]:
def build_classifier_prompt(subject, sender, body):
    return f"""
# ROLE
You are an expert support email classifier for a B2B SaaS company.
You have 10 years of experience triaging customer support emails.

# TASK  
Classify the incoming email into exactly one primary category,
one urgency level, and one sub-category where applicable.

# CATEGORIES
Primary (pick ONE):
- Billing
- Technical
- Feature Request
- Spam
- Other

Urgency (pick ONE):
- High    (customer cannot use the product)
- Medium  (issue exists but workaround available)
- Low     (question or minor inconvenience)

Billing Sub-category (only if Primary = Billing):
- Refund Request
- Failed Payment
- Pricing Question
- Invoice Issue
- Not Applicable

# RULES
- RETURN ONLY valid JSON. No explanation. No markdown.
- If the email is in any language other than English, still classify it.
- If you cannot determine the category, use "Other" and urgency "Low".

# OUTPUT FORMAT
Return ONLY this JSON structure, nothing else:
{{
  "category": "[Primary Category]",
  "urgency": "[Urgency Level]",
  "billing_sub": "[Sub-category or Not Applicable]"
}}

# EMAIL TO CLASSIFY
Subject: {subject}
From: {sender}
Body: {body}
"""

prompt = build_classifier_prompt(
    subject="Urgent: Team can't login - product demo in 2 hours!",
    sender="cto@bigclient.com",
    body="Our entire team is locked out. SSO is broken. We have a board demo in 2 hours. This is critical."
)

def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

call_llm(prompt)

'{\n  "category": "Technical",\n  "urgency": "High",\n  "billing_sub": "Not Applicable"\n}'

---
## Task 2 — Add `language` Field to Output

In [5]:
# Add language detection to the output
# Only change: OUTPUT FORMAT section + JSON structure

STRUCTURED_PROMPT_V2 = """CRITICAL: Return ONLY valid JSON. No explanation. No markdown.

# ROLE
You are an expert support email classifier for a B2B SaaS company.

# CATEGORIES
Primary: Billing | Technical | Feature Request | Spam | Other
Urgency: High | Medium | Low
Billing Sub: Refund Request | Failed Payment | Pricing Question | Invoice Issue | Not Applicable
Language: English | Hindi | Hinglish | Other

# RULES
- RETURN ONLY valid JSON.
- Hinglish = mix of Hindi and English in same sentence.
- Classify correctly regardless of language.

# OUTPUT FORMAT — Return ONLY this JSON:
{{"category": "...", "urgency": "...", "billing_sub": "...", "language": "..."}}

# EMAIL
Subject : {subject}
From    : {sender}
Body    : {body}"""


def classify_with_language(email):
    prompt = STRUCTURED_PROMPT_V2.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return json.loads(response.choices[0].message.content.strip())


print("=" * 60)
print("WITH LANGUAGE DETECTION")
print("=" * 60)

for email in test_emails:
    result = classify_with_language(email)
    print(f"\nSubject  : {email['subject']}")
    print(f"Category : {result['category']}")
    print(f"Urgency  : {result['urgency']}")
    print(f"Language : {result['language']}")
    time.sleep(1)

print()
print("Lesson: Adding a field = 2 changes only:")
print("  1. Add to # CATEGORIES section")
print("  2. Add to # OUTPUT FORMAT JSON")
print("  Function code = zero changes!")

WITH LANGUAGE DETECTION

Subject  : Payment refund request
Category : Billing
Urgency  : High
Language : Hinglish

Subject  : App crash ho raha hai
Category : Technical
Urgency  : High
Language : Hinglish

Subject  : Billing mein koi samasya hai
Category : Billing
Urgency  : High
Language : Hinglish

Lesson: Adding a field = 2 changes only:
  1. Add to # CATEGORIES section
  2. Add to # OUTPUT FORMAT JSON
  Function code = zero changes!


---
## Task 3 — Break It: Remove the RULES Section

In [6]:
# What happens when you remove # RULES?
# Run the same emails — see what breaks

PROMPT_NO_RULES = """# ROLE
You are an expert support email classifier for a B2B SaaS company.

# TASK
Classify the email into exactly one primary category and urgency level.

# CATEGORIES
Primary: Billing | Technical | Feature Request | Spam | Other
Urgency: High | Medium | Low
Billing Sub: Refund Request | Failed Payment | Pricing Question | Invoice Issue | Not Applicable

# OUTPUT FORMAT
{{"category": "...", "urgency": "...", "billing_sub": "..."}}

# EMAIL
Subject : {subject}
From    : {sender}
Body    : {body}"""


def classify_no_rules(email):
    prompt = PROMPT_NO_RULES.format(
        subject=email['subject'],
        sender=email['sender'],
        body=email['body']
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    # Sometimes returns markdown — try to clean
    raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except:
        return {"raw": raw, "parse_error": True}


# Compare: with rules vs without rules
print("=" * 60)
print("WITH RULES vs WITHOUT RULES")
print("=" * 60)

for email in test_emails:
    with_rules    = classify_structured(email)
    time.sleep(0.5)
    without_rules = classify_no_rules(email)

    print(f"\nSubject       : {email['subject']}")
    print(f"With rules    : {with_rules}")
    print(f"Without rules : {without_rules}")
    time.sleep(0.5)

print()
print("What breaks without RULES:")
print("  - Model may add markdown fences (```json)")
print("  - Model may add explanation text before JSON")
print("  - Hindi emails may get wrong language handling")
print("  - Urgency may be inconsistent")
print()
print("Lesson: RULES section = your constraints = production safety")

WITH RULES vs WITHOUT RULES

Subject       : Payment refund request
With rules    : {"category": "Billing", "urgency": "High", "billing_sub": "Refund Request"}
Without rules : {'category': 'Billing', 'urgency': 'High', 'billing_sub': 'Refund Request'}

Subject       : App crash ho raha hai
With rules    : {"category": "Technical", "urgency": "High", "billing_sub": "Not Applicable"}
Without rules : {'category': 'Technical', 'urgency': 'High', 'billing_sub': 'Not Applicable'}

Subject       : Billing mein koi samasya hai
With rules    : {"category": "Billing", "urgency": "High", "billing_sub": "Refund Request"}
Without rules : {'category': 'Billing', 'urgency': 'High', 'billing_sub': 'Refund Request'}

What breaks without RULES:
  - Model may add markdown fences (```json)
  - Model may add explanation text before JSON
  - Hindi emails may get wrong language handling
  - Urgency may be inconsistent

Lesson: RULES section = your constraints = production safety


---
## Chapter 4 Summary

| Section | Purpose | Without it |
|---------|---------|------------|
| `# ROLE` | Sets persona | Generic answers |
| `# TASK` | Describes goal | Model guesses what you want |
| `# CATEGORIES` | Controlled vocabulary | Model invents labels |
| `# RULES` | Edge case handling | Format breaks, markdown added |
| `# OUTPUT FORMAT` | Machine-parseable JSON | Free text, unparseable |
| `# EMAIL` | Structured input | Subject/sender ignored |

### What's Next?
Chapter 5 — **Loss in the Middle**  
Your structured prompt works for short emails.  
But a 3-page email thread breaks everything. Here's why.